
# Cuando el camino más corto no es el mejor

Ruta de evacuación en una zona inundada 
Equipo: CafeSmart
Integrantes:
Morales Mendez Shryley Nicoles
Jimenez Andrade Monserrat
Tejero Hernandez Rubi
Sarao Morales Rubi

Un agente debe ir de un punto inicial a un refugio sobre un terreno con celdas secas, de inundación leve / media / alta (transitables pero riesgosas) y bloqueadas. Se comparan cuatro estrategias (BFS, Dijkstra, A*, Voraz) con ejecuciones reales.

Índice: 
· 1 Modelo 
· 2 Algoritmos 
· 3 Validador 
· 4 Escenario y visualización
 · 5 Comparación 
 · 6 Pruebas (T1–T5) 
 · 7 Prueba adicional (T6) 
 · 8 Decisiones y declaración de IA

## 1. Modelo del terreno
Cada celda tiene un nivel: 0 seco, 1 leve, 2 media, 3 alta, 4 bloqueado.

- **Bloqueo = restricción dura:** las celdas nivel 4 nunca se expanden.
- **Riesgo = costo blando:** costo de entrar a una celda = `1 + λ · riesgo(nivel)`, con riesgo 0, 1, 3, 10. El "1" es el esfuerzo de dar un paso; el resto penaliza el peligro. La penalización no es lineal para que una celda alta pese más que varias leves.
- **Movimiento en 4 direcciones** (sin diagonales, para evitar "cortar esquinas" entre bloqueos).
- **Costos ≥ 1**, así la distancia Manhattan nunca sobreestima el costo restante (heurística admisible para A*).

In [1]:
import heapq, time, random
from collections import deque

# Niveles de terreno
DRY, LOW, MID, HIGH, BLOCKED = 0, 1, 2, 3, 4
# Penalización de riesgo por entrar a una celda (sumada al costo base 1)
RISK_W = {DRY: 0, LOW: 1, MID: 3, HIGH: 10}
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 4-conectividad


def cell_cost(level, lam=1.0, w=RISK_W):
    """Costo de ENTRAR a una celda: 1 + lam * riesgo. None si bloqueada."""
    if level == BLOCKED:
        return None
    return 1.0 + lam * w[level]


def neighbors(grid, pos):
    r, c = pos
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != BLOCKED:
            yield (nr, nc)


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def path_cost(grid, path, lam=1.0, w=RISK_W):
    return sum(cell_cost(grid[r][c], lam, w) for r, c in path[1:])


def _rebuild(parent, goal):
    p, cur = [], goal
    while cur is not None:
        p.append(cur)
        cur = parent[cur]
    return p[::-1]


def _result(grid, path, expanded, pushed, t0, lam, w):
    return {
        "path": path,
        "cost": None if path is None else path_cost(grid, path, lam, w),
        "steps": None if path is None else len(path) - 1,
        "expanded": expanded,
        "generated": pushed,
        "time_ms": (time.perf_counter() - t0) * 1000,
    }

## 4. Escenario base

Un muro bloqueado (columnas 6–8) divide el mapa. Hay un **atajo** por la fila 7 con 3 celdas de inundación alta y un **rodeo seco** por la fila 14. Inicio (7,1), refugio (7,14) (fila, columna).

In [2]:
# ---------------- Escenarios ----------------
def base_scenario():
    """16x16. Muro inundado (bloqueado) en cols 6-8 con dos pasos:
    atajo por fila 7 (riesgo ALTO) y rodeo seco por fila 14."""
    n = 16
    g = [[DRY] * n for _ in range(n)]
    for r in range(n):
        for c in (6, 7, 8):
            g[r][c] = BLOCKED
    for c in (6, 7, 8):
        g[7][c] = HIGH       # atajo peligroso
        g[14][c] = DRY       # rodeo seguro
    # obstaculo realista: zonas de inundación leve/media
    for r in range(2, 5):
        for c in range(1, 4):
            g[r][c] = LOW
    for r in range(10, 13):
        for c in range(10, 14):
            g[r][c] = MID
    for r in range(3, 6):
        for c in range(11, 14):
            g[r][c] = LOW
    return g, (7, 1), (7, 14)


def random_grid(n, rng, p_block=0.2, p_risk=0.35):
    g = [[DRY] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            x = rng.random()
            if x < p_block: g[r][c] = BLOCKED
            elif x < p_block + p_risk: g[r][c] = rng.choice([LOW, MID, HIGH])
    g[0][0] = DRY; g[n - 1][n - 1] = DRY
    return g

## 2. Estrategias de búsqueda
| Estrategia | Optimiza | ¿Ve el riesgo? | Garantía |
|---|---|---|---|
| BFS | número de pasos | No (solo bloqueos) | mínimo de pasos |
| Dijkstra | costo acumulado g(n) | Sí | costo óptimo |
| A* | g(n) + h(n), h = Manhattan | Sí | costo óptimo (h admisible) |
| Voraz | solo h(n) | No | ninguna |

Medida de trabajo: **nodos expandidos** (procesados), **generados** (insertados en la cola) y tiempo.

In [3]:
def bfs(grid, start, goal, lam=1.0, w=RISK_W):
    """Menos pasos; ignora el riesgo (sólo respeta bloqueos)."""
    t0 = time.perf_counter()
    if grid[start[0]][start[1]] == BLOCKED or grid[goal[0]][goal[1]] == BLOCKED:
        return _result(grid, None, 0, 0, t0, lam, w)
    parent, q, exp, gen = {start: None}, deque([start]), 0, 1
    while q:
        cur = q.popleft(); exp += 1
        if cur == goal:
            return _result(grid, _rebuild(parent, goal), exp, gen, t0, lam, w)
        for nb in neighbors(grid, cur):
            if nb not in parent:
                parent[nb] = cur; q.append(nb); gen += 1
    return _result(grid, None, exp, gen, t0, lam, w)

In [4]:
def _best_first(grid, start, goal, lam, w, use_g, h_scale=1.0, use_h=True):
    """Núcleo común de Dijkstra (g), A* (g+h) y Voraz (h)."""
    t0 = time.perf_counter()
    if grid[start[0]][start[1]] == BLOCKED or grid[goal[0]][goal[1]] == BLOCKED:
        return _result(grid, None, 0, 0, t0, lam, w)
    g = {start: 0.0}
    parent = {start: None}
    h = (lambda p: h_scale * manhattan(p, goal)) if use_h else (lambda p: 0.0)
    pq = [(h(start), 0, start)]
    tie, closed, exp, gen = 0, set(), 0, 1
    while pq:
        _, _, cur = heapq.heappop(pq)
        if cur in closed:
            continue
        closed.add(cur); exp += 1
        if cur == goal:
            return _result(grid, _rebuild(parent, goal), exp, gen, t0, lam, w)
        for nb in neighbors(grid, cur):
            ng = g[cur] + cell_cost(grid[nb[0]][nb[1]], lam, w)
            if use_g:
                better = nb not in g or ng < g[nb]
            else:
                better = nb not in g
            if better:
                g[nb] = ng; parent[nb] = cur
                pri = (ng if use_g else 0.0) + h(nb)
                tie += 1
                heapq.heappush(pq, (pri, tie, nb)); gen += 1
    return _result(grid, None, exp, gen, t0, lam, w)


def dijkstra(grid, start, goal, lam=1.0, w=RISK_W):
    return _best_first(grid, start, goal, lam, w, use_g=True, use_h=False)


def astar(grid, start, goal, lam=1.0, w=RISK_W, h_scale=1.0):
    return _best_first(grid, start, goal, lam, w, use_g=True, h_scale=h_scale)


def greedy(grid, start, goal, lam=1.0, w=RISK_W):
    """Voraz: sólo heurística, ignora el costo acumulado y el riesgo."""
    return _best_first(grid, start, goal, lam, w, use_g=False)


ALGOS = {"BFS": bfs, "Dijkstra": dijkstra, "A*": astar, "Voraz": greedy}